# Heston stochastic volatility

How the Heston model works, step by step, on TICKER1:

1. **why a stochastic vol**: what local vol cannot do;
2. **the model**: the dynamics, what each of the five parameters does to the smile, the term structures it can produce;
3. **the closed form**: characteristic function and COS method;
4. **the calibration, by hand**: targets, weights, least squares, fit quality, identifiability, sensitivities through
   the calibration, compared with `HestonModel.from_equity`;
5. **what the model implies**: the forward smile, against local vol;
6. **Monte Carlo**: the QE scheme, closed form vs simulation, greeks by AAD;
7. **the limits** of the model.

Docs: *Models and Pricing* (Models: Heston), *Architecture* (Models). Read after `local_vol.ipynb`.

In [ ]:
import os
import time
from datetime import timedelta
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorquant as tq
from scipy.optimize import brentq, least_squares
from scipy.stats import norm

## The calibration in a nutshell

**Input**: the same implied vol surface as for local vol, $\sigma_{ij}$ (8 maturities $\times$ 9 strikes = 72 quotes).
**Output**: **five numbers**, $v_0, \kappa, \theta, \xi, \rho$.

The difference with local vol is fundamental. Local vol has a free function $\sigma_{\mathrm{loc}}(t, S)$, as many degrees of
freedom as needed, and Dupire *computes* it: the fit is exact. Heston has five parameters for 72 quotes, so in general
no choice reprices them all: the calibration is an **optimisation**, which looks for the five parameters whose smiles
are closest to the market ones, and the fit has errors.

The four operations:

1. **Targets**: every quote becomes the price of a call (per unit of forward), and every maturity gets a weight,
   $1 / \text{ATM vega}$, so that a price error reads as an implied vol error.
2. **Model prices in closed form**: for given parameters, the 72 Heston prices from the characteristic function
   (COS method), fast enough to be called hundreds of times.
3. **Least squares**: minimise the sum of the squared weighted errors over the five parameters, within bounds.
4. **Report**: the errors per maturity, the Feller condition, the parameters on a bound. The sensitivities of the
   parameters to the quotes come from the optimality condition (implicit function theorem), so greeks flow through
   the calibration.

```
quotes (T_i, K_j, vol_ij) ──1──> targets c_ij (Black calls) and weights 1 / ATM vega_i
params (v0, kappa, theta, xi, rho) ──2──> Heston prices c_ij(params)  (characteristic function + COS)
                             ──3──> min sum_ij [ (c_ij(params) - c_ij) / vega_i ]^2  ──> params*
                             ──4──> errors per maturity, Feller, dparams / dquotes
```

The model and its parameters are section 2, the closed form section 3, the calibration step by step section 4.

## 1. Why a stochastic vol
Local vol reprices today's surface exactly, but its vol is a **deterministic function** of time and spot
(`local_vol.ipynb`, section 6): the future smiles are flat, the vol has no randomness of its own, the smile moves the
wrong way with the spot. In the market the vol of a stock moves on its own: it rises in sell-offs (negative correlation
with the spot, the *leverage effect*), it comes in clusters and it reverts to a long-run level.

Heston (1993) makes the **variance** a random process, mean-reverting and correlated with the spot. It is the standard
stochastic vol model in equity because, despite the second factor, vanillas have a semi-closed form.

In [ ]:
mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")   # latest snapshot
EVALUATION_DATE = tq.Settings.evaluation_date = mkt.evaluation_date
TICKER = "TICKER1"
eq = mkt.get_equity(TICKER)
vs = eq.vol_surface
S0 = float(eq.spot)
T_NODES = np.array(vs.maturity, dtype=float)
K_NODES = np.array(vs.strike, dtype=float)
IV = vs.volatility_matrix.numpy()                      # [maturities, strikes]
F_NODES = np.array([float(eq.forward(float(T))) for T in T_NODES])
K_MKT = K_NODES[None, :] / F_NODES[:, None]            # strikes per unit forward [maturities, strikes]


def date_of(T):
    return EVALUATION_DATE + timedelta(days=max(1, int(round(T * 365))))


print(f"{TICKER} spot {S0:.2f}, maturities {np.round(T_NODES, 2)}")
print("implied vols (%), maturity x strike / spot:")
print(pd.DataFrame(100 * IV, index=np.round(T_NODES, 2), columns=[f"{k:.0%}" for k in K_NODES / S0]).round(1).to_string())

In [ ]:
# Black-Scholes per unit forward (undiscounted) and its inversion, used throughout
def bs_call(k, T, s):
    sd = s * np.sqrt(T)
    d1 = -np.log(k) / sd + 0.5 * sd
    return norm.cdf(d1) - k * norm.cdf(d1 - sd)


def implied(call, k, T):
    # implied vol of an undiscounted call per unit forward, inverted on the OTM side (put below the forward)
    otm = call - (1.0 - k) if k < 1 else call
    f = (lambda s: bs_call(k, T, s) - (1.0 - k) - otm) if k < 1 else (lambda s: bs_call(k, T, s) - otm)
    try:
        return brentq(f, 1e-4, 4.0)
    except ValueError:
        return np.nan


def heston_smile(params, k, T):
    # Heston implied vols at strikes k (per unit forward) for one maturity
    k = np.atleast_1d(np.asarray(k, dtype=float))
    c = tq.heston_price(k[None, :], [T], *params).numpy()[0]
    return np.array([implied(ci, ki, T) for ci, ki in zip(c, k)])

## 2. The model
On the forward ratio $X = S / F(t)$ (rates, repo and dividends stay in the forward, as for local vol):
$$\frac{\mathrm{d}X_t}{X_t} = \sqrt{v_t}\,\mathrm{d}W^1_t,\qquad
\mathrm{d}v_t = \kappa\,(\theta - v_t)\,\mathrm{d}t + \xi\,\sqrt{v_t}\,\mathrm{d}W^2_t,\qquad
\mathrm{d}\langle W^1, W^2\rangle_t = \rho\,\mathrm{d}t .$$
The instantaneous vol is $\sqrt{v_t}$; the variance $v_t$ is a **square-root (CIR) process**:

| parameter | name | in the dynamics | on the smile |
|---|---|---|---|
| $v_0$ | initial variance | today's instantaneous variance | level of the short maturities |
| $\theta$ | long-run variance | the level $v$ reverts to | level of the long maturities |
| $\kappa$ | mean reversion | speed of the reversion (half-life $\ln 2 / \kappa$) | how fast the short-term features fade with the maturity |
| $\xi$ | vol of vol | size of the random moves of $v$ | curvature (fat tails on both sides), and the size of the skew |
| $\rho$ | spot–vol correlation | $v$ rises when the spot falls if $\rho < 0$ | sign and slope of the skew |

Two properties used below:
- **expected variance**: $\mathbb{E}[v_t] = \theta + (v_0 - \theta)\,e^{-\kappa t}$, from $v_0$ to $\theta$ with speed $\kappa$;
- **Feller condition**: $2\kappa\theta \ge \xi^2$ keeps $v$ strictly positive; when it fails (common in equity calibrations)
  the variance touches zero and bounces back. Not an arbitrage, but it matters for the simulation (section 6).

The parameters below are the calibrated ones (how: section 4).

In [ ]:
t0 = time.time()
model = tq.HestonModel.from_equity(eq)
print(f"calibration: {time.time() - t0:.1f} s (the first call also compiles the TensorFlow graphs)")
P = [float(v) for v in model.parameters.values()]                  # v0, kappa, theta, xi, rho
v0, kappa, theta, xi, rho = P
print(pd.Series(model.parameters).map(float).round(4).to_string())
print(f"vol: today sqrt(v0) = {np.sqrt(v0):.1%}, long run sqrt(theta) = {np.sqrt(theta):.1%}; "
      f"half-life of the variance ln2/kappa = {np.log(2) / kappa:.1f} years")
print(f"Feller 2 kappa theta - xi^2 = {model.feller:.3f}  ({'satisfied' if model.feller >= 0 else 'violated: v reaches 0'})")

In [ ]:
# variance paths (QE scheme, section 6), and E[v_t] against the closed form
grid = np.arange(1, 5 * 52 + 1) / 52                              # weekly to 5Y
dw = tf.random.stateless_normal([20_000, len(grid), 2], seed=[1, 0], dtype=tf.float64)
X, V = (a.numpy() for a in model.simulate(grid, dw))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].plot(grid, np.sqrt(V[:30].T), color="steelblue", alpha=0.4, lw=0.7)
axes[0].plot(grid, np.sqrt(V.mean(axis=0)), "k", lw=2, label="sqrt of the MC mean of v")
axes[0].plot(grid, np.sqrt(theta + (v0 - theta) * np.exp(-kappa * grid)), "r--", lw=1.5, label="sqrt(E[v_t]) closed form")
axes[0].set(xlabel="t", ylabel="sqrt(v)", title="Instantaneous vol paths")
axes[0].legend(fontsize=8)
axes[1].hist(V[:, 51], bins=120, range=(0, 0.6), density=True, color="steelblue")
axes[1].set(xlabel="v at 1Y", title=f"Variance at 1Y: {np.mean(V[:, 51] < 1e-3):.0%} of the paths below 0.001")
for ax in axes:
    ax.grid(alpha=0.3)
plt.show()

With Feller violated the variance distribution has a lot of mass near zero and a long right tail: the vol of vol is
large compared with the pull to the mean. The tail produces the fat tails of the stock (the smile curvature), the
correlation tilts them to the downside (the skew).

### What each parameter does to the smile
One parameter at a time moved around the calibrated value, the others fixed; 3M and 2Y smiles (closed form, section 3).

In [ ]:
k_plot = np.linspace(0.6, 1.4, 41)
names = ["v0", "kappa", "theta", "xi", "rho"]
moves = {"v0": (0.5, 1.5), "kappa": (0.3, 3.0), "theta": (0.5, 1.5), "xi": (0.3, 1.5), "rho": (0.0, 2.0)}   # factors
fig, axes = plt.subplots(2, 5, figsize=(17, 6), sharey="row")
for row, T in enumerate((0.25, 2.0)):
    for col, n in enumerate(names):
        ax = axes[row, col]
        for f, style in ((moves[n][0], ":"), (1.0, "-"), (moves[n][1], "--")):
            p = list(P)
            p[col] = max(-0.99, min(0.99, P[col] * f)) if n == "rho" else P[col] * f
            ax.plot(k_plot, 100 * heston_smile(p, k_plot, T), style, label=f"{n} = {p[col]:.3g}")
        ax.set(title=f"{n}, T = {T}", xlabel="k = K / F")
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)
    axes[row, 0].set_ylabel("implied vol (%)")
plt.tight_layout()
plt.show()

Reading:
- **$v_0$** moves the short smile most; with the slow reversion here (half-life 1.9 years) it still moves the 2Y one.
  **$\theta$** barely touches the 3M smile and moves the 2Y one.
- **$\kappa$**: a faster reversion pulls the variance back to $\theta$ sooner, so the vol of vol and the correlation act for a
  shorter time: the 2Y smile is flatter and higher (closer to $\sqrt\theta$, see the term structure below). At 3M, no effect.
- **$\xi$**: more vol of vol, more curvature (both wings up) and, with $\rho < 0$, more skew; it also clearly **lowers the ATM
  vol**, more at long maturities (next paragraph).
- **$\rho$**: $\rho = 0$ gives a symmetric smile (a smile, not a skew); more negative, steeper skew; at 2Y it also lowers the ATM vol
  by about a vol point.

### ATM term structure: the average vol, not the average variance
Take $\rho = 0$ for the intuition. Given the whole path of $v$, the stock is lognormal with total variance $\int_0^T v_t\,\mathrm{d}t$,
so the call is the Black–Scholes price averaged over the realised **average variance** $\bar v_T = \frac1T\int_0^T v_t\,\mathrm{d}t$
(Hull and White's mixing argument):
$$c = \mathbb{E}\big[c_{\mathrm{BS}}(\bar v_T)\big].$$
At the money the Black–Scholes price is almost **linear in the vol**, $c_{\mathrm{BS}} \approx 0.4\,\sigma\sqrt T$, so
$$\sigma_{\mathrm{ATM}}(T) \approx \mathbb{E}\big[\sqrt{\bar v_T}\,\big] \;<\; \sqrt{\mathbb{E}[\bar v_T]} = \sqrt{\theta + (v_0 - \theta)\,\frac{1 - e^{-\kappa T}}{\kappa T}} .$$
The gap (Jensen's inequality) grows with the dispersion of $\bar v_T$, i.e. with the vol of vol and the maturity. Here
$\sqrt{v_0} \approx \sqrt\theta \approx 35\%$, so the expected variance is flat, yet the market ATM term structure falls from 32% to 25%:
the calibration produces the fall with a large $\xi$, not with $v_0 > \theta$. Below, the three curves (the middle one from
the simulated variance paths above).

In [ ]:
T_plot = np.linspace(0.02, 5.2, 80)
atm_heston = [heston_smile(P, [1.0], T)[0] for T in T_plot]
atm_avg = np.sqrt(theta + (v0 - theta) * (1 - np.exp(-kappa * T_plot)) / (kappa * T_plot))
v_bar = np.cumsum(V, axis=1) / np.arange(1, len(grid) + 1)              # running average of v on the weekly paths
atm_mix = np.sqrt(v_bar).mean(axis=0)
atm_rho0 = [heston_smile([v0, kappa, theta, xi, 0.0], [1.0], T)[0] for T in T_plot]
atm_mkt = [float(vs.volatility(float(eq.forward(float(T))), float(T))) for T in T_NODES]
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(T_plot, 100 * atm_avg, "--", label="sqrt(E[average v]): no vol of vol")
ax.plot(grid, 100 * atm_mix, ":", lw=2, label="E[sqrt(average v)] (MC)")
ax.plot(T_plot, 100 * np.array(atm_rho0), "-.", label="Heston ATM with rho = 0")
ax.plot(T_plot, 100 * np.array(atm_heston), label="Heston ATM (closed form)")
ax.plot(T_NODES, 100 * np.array(atm_mkt), "ko", ms=5, label="market ATM")
ax.set(xlabel="T", ylabel="vol (%)", title="ATM term structure")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

- With $\rho = 0$ the Heston ATM vol is $\mathbb{E}[\sqrt{\bar v_T}]$, as the argument says (up to the small curvature of the ATM price in
  the vol at long maturities): most of the fall comes from the vol of vol.
- The negative correlation lowers it by a further 1–2 vol points at long maturities: the argument still works with
  $\rho \ne 0$, but each path then also shifts the forward and reduces the variance by $(1 - \rho^2)$.

So in Heston the **ATM term structure is set by $v_0$, $\theta$, $\kappa$, the vol of vol and the correlation together**, one more
reason why the parameters are entangled (section 4, step 5).

### The skew term structure: what five constant parameters cannot do
The ATM skew $\partial\sigma_{\mathrm{imp}}/\partial y$ ($y = \ln K/F$) of Heston has a finite limit at short maturities,
$$\frac{\partial\sigma_{\mathrm{imp}}}{\partial y}\Big|_{y = 0,\ T \to 0} = \frac{\rho\,\xi}{4\sqrt{v_0}},$$
(Gatheral, 2006, ch. 3) and once $\kappa T \gg 1$ it decays like $1/T$, because the mean reversion averages the variance out
(here $\kappa$ is small, so the decay is slow over 5 years). The equity market skew instead **grows** as the maturity
shrinks (roughly like $T^{-1/2}$). One set of parameters cannot do both: a steep short skew needs a large $\xi$ (or a
small $v_0$), which makes the long skew too steep. This is the main source of the calibration errors (section 4).

In [ ]:
def atm_skew(vol_fn, h=0.02):
    return (vol_fn(h) - vol_fn(-h)) / (2 * h)


rows = []
for T in (1 / 52, *T_NODES):
    F_T = float(eq.forward(float(T)))
    mkt_skew = atm_skew(lambda y: float(vs.volatility(F_T * np.exp(y), float(T)))) if T >= T_NODES[0] else np.nan
    hes_skew = atm_skew(lambda y: heston_smile(P, [np.exp(y)], T)[0])
    rows.append({"T": round(T, 3), "market skew": mkt_skew, "Heston skew": hes_skew})
print(pd.DataFrame(rows).set_index("T").round(3).to_string())
print(f"\nshort-maturity limit rho xi / (4 sqrt v0) = {rho * xi / (4 * np.sqrt(v0)):.3f}")

The calibration compromises: the Heston skew is too flat at the short maturities and too steep at the long ones, close
to the market only around 1Y.

## 3. The closed form
**Why there is one.** Heston is an *affine* model: the drift and the variance of $(\ln X, v)$ are linear in $v$. For such
models the characteristic function of $\ln X_T$ is exponential-affine in the initial variance,
$$\varphi(u) = \mathbb{E}\big[e^{iu\ln X_T}\big] = \exp\big(C(T, u) + D(T, u)\,v_0\big),$$
and $C$, $D$ solve ordinary differential equations (Riccati) that have a closed-form solution. In the form that stays
continuous in $u$ (Albrecher et al., 2007, the "little Heston trap"), with $b = \kappa - \rho\xi iu$, $d = \sqrt{b^2 + \xi^2(iu + u^2)}$,
$g = (b - d)/(b + d)$:
$$C = \frac{\kappa\theta}{\xi^2}\Big[(b - d)\,T - 2\ln\frac{1 - g e^{-dT}}{1 - g}\Big],\qquad
D = \frac{b - d}{\xi^2}\,\frac{1 - e^{-dT}}{1 - g e^{-dT}} .$$

**From the characteristic function to prices: the COS method** (Fang and Oosterlee, 2008).
1. Truncate $x = \ln X_T$ to an interval $[a, b]$ that holds all the mass (here mean $\pm$ 30 standard deviations, from the
   cumulants).
2. On $[a, b]$ the density is a cosine series, and its coefficients are the characteristic function at the frequencies
   $u_n = n\pi / (b - a)$:
   $$f(x) \approx \frac{2}{b - a}\,{\sum_{n=0}^{N-1}}{}'\ \mathrm{Re}\big[\varphi(u_n)\,e^{-iu_n a}\big]\cos\big(u_n (x - a)\big)$$
   (the prime: the first term halved), because $\varphi(u_n) = \int e^{iu_n x} f(x)\,\mathrm{d}x$ is almost the integral that defines the
   cosine coefficient.
3. The price is $\int \text{payoff}(x)\,f(x)\,\mathrm{d}x$: the cosine coefficients of the payoff are known in closed form, so the price
   is a sum of $N$ terms. The put (bounded payoff) is computed this way, the call by put–call parity.

The density converges exponentially in $N$ for a smooth density: a few hundred terms give machine precision. Below, the
1Y density of $\ln X$ by COS against the simulated paths, and the convergence of the price in $N$.

In [ ]:
T1 = 1.0
c1, c2 = (float(np.squeeze(v)) for v in tq.heston_cumulants(T1, *P))
a, b = c1 - 12 * np.sqrt(c2), c1 + 12 * np.sqrt(c2)
N = 256
u = np.arange(N) * np.pi / (b - a)
phi = tq.heston_characteristic_function(u, T1, *P).numpy()
coef = np.real(phi * np.exp(-1j * u * a))
coef[0] *= 0.5
x = np.linspace(-1.5, 1.0, 400)
density = 2 / (b - a) * (coef[None, :] * np.cos(u[None, :] * (x[:, None] - a))).sum(axis=1)

lnX1 = np.log(X[:, 51])                                         # the simulated paths of section 2 (1Y = week 52)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].hist(lnX1, bins=150, range=(-1.5, 1.0), density=True, color="lightsteelblue", label="QE Monte Carlo")
axes[0].plot(x, density, "k", lw=1.5, label=f"COS, N = {N}")
sd = np.sqrt(c2)
axes[0].plot(x, norm.pdf(x, c1, sd), "r--", lw=1, label="normal, same mean and variance")
axes[0].set(xlabel="ln X at 1Y", title="Density of ln X_1")
axes[0].legend(fontsize=8)

ref = {T: tq.heston_price(np.array([[0.8, 1.0, 1.2]]), [T], *P, n_terms=4096).numpy()[0] for T in (1 / 12, 1.0)}
Ns = [8, 16, 32, 64, 128, 256, 512, 1024]
for T, style in ((1 / 12, "o-"), (1.0, "s-")):
    errs = [np.abs(tq.heston_price(np.array([[0.8, 1.0, 1.2]]), [T], *P, n_terms=n).numpy()[0] - ref[T]).max() for n in Ns]
    axes[1].semilogy(Ns, np.maximum(errs, 1e-17), style, label=f"T = {T:.2f}")
axes[1].set(xlabel="N (cosine terms)", ylabel="max price error (per unit forward)",
            title="COS convergence (strikes 80/100/120%, range +/- 30 sd)")
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.3)
plt.show()

The density is negatively skewed and fat-tailed compared with the normal of the same variance: that is the smile. The
COS error falls exponentially until machine precision; the short maturity needs more terms (its density is narrower
on the same $\pm 30$ standard deviations). The library prices with 1024 terms.

## 4. Calibration, step by step

### 4.0 What is fitted, to what, and how
- **What**: the five parameters, constant in time.
- **To what**: all 72 quotes at once, as **call prices per unit of forward**, $c_{ij} = \mathbb{E}[(X_{T_i} - k_{ij})^+]$, $k_{ij} = K_j / F(T_i)$.
  Prices, not implied vols: the model gives prices, and inverting 72 implied vols at every iteration would be slow
  and not smooth.
- **Weights**: each maturity is divided by its ATM vega, $\mathcal V_i = \partial c / \partial\sigma$ at the money. A price error $\delta c$ at the money is
  then $\delta c / \mathcal V_i \approx \delta\sigma$, an **implied vol error**, and all maturities weigh alike (a 1M ATM vega is about $\sqrt{1/12}$ of a 1Y one).
  In the wings the residual is $\delta\sigma \times \mathcal V_{ij}/\mathcal V_i$: the wings count by their vega, less than the centre.
- **Objective**:
  $$\min_{v_0, \kappa, \theta, \xi, \rho}\ \sum_{i, j}\Big(\frac{c^{\mathrm{Heston}}_{ij} - c^{\mathrm{mkt}}_{ij}}{\mathcal V_i}\Big)^2$$
  within bounds: $v_0, \theta \in [10^{-4}, 1]$, $\kappa \in [0.01, 20]$, $\xi \in [0.01, 5]$, $\rho \in [-0.99, 0.99]$.
- **Start**: $v_0$ = the shortest ATM variance, $\theta$ = the average ATM variance, $\kappa = 1.5$, $\xi = 0.5$, $\rho = -0.6$.
- **Optimiser**: trust-region least squares (Levenberg–Marquardt type, with bounds), Jacobian by finite differences. It
  is a local search: the start matters (section 4.4).

### Step 1: targets and weights

In [ ]:
MARKET = bs_call(K_MKT, T_NODES[:, None], IV)                           # Black calls per unit forward
i_atm = np.argmin(np.abs(np.log(K_MKT)), axis=1)                        # the node nearest to the forward
ATM = IV[np.arange(len(T_NODES)), i_atm]
VEGA = np.sqrt(T_NODES) * np.exp(-ATM ** 2 * T_NODES / 8) / np.sqrt(2 * np.pi)   # dc/dsigma at the money
print(pd.DataFrame({"T": T_NODES, "ATM vol": ATM, "ATM vega": VEGA, "weight 1/vega": 1 / VEGA}).round(4).to_string(index=False))

### Step 2: the objective
The residuals, one per quote, with a cheaper COS during the search (256 terms on $\pm 12$ standard deviations: about
$10^{-7}$, plenty for a search; the library polishes the optimum at full precision).

In [ ]:
NAMES = ["v0", "kappa", "theta", "xi", "rho"]
LOWER = np.array([1e-4, 1e-2, 1e-4, 1e-2, -0.99])
UPPER = np.array([1.0, 20.0, 1.0, 5.0, 0.99])


def residuals(x, rows=slice(None)):
    model_prices = tq.heston_price(K_MKT[rows], T_NODES[rows], *x, n_terms=256, width=12.0).numpy()
    return ((model_prices - MARKET[rows]) / VEGA[rows, None]).ravel()


x_start = np.array([ATM[0] ** 2, 1.5, np.mean(ATM ** 2), 0.5, -0.6])
for name, x in (("start", x_start), ("library optimum", np.array(P))):
    r = residuals(x)
    print(f"{name:16s}: rms residual {100 * np.sqrt(np.mean(r ** 2)):.3f} vol points, objective {0.5 * np.sum(r ** 2):.2e}")

### Step 3: the least squares

In [ ]:
t0 = time.time()
fit = least_squares(residuals, x_start, bounds=(LOWER, UPPER), method="trf", x_scale="jac", jac="2-point",
                    xtol=1e-12, ftol=1e-12, gtol=1e-12)
print(f"{fit.nfev} evaluations, {time.time() - t0:.1f} s, status: {fit.message}")
print(pd.DataFrame({"by hand": fit.x, "HestonModel": P}, index=NAMES).round(5).to_string())

The same optimum as the library (which then polishes it on the full-precision COS and makes it differentiable, step 6).

### Step 4: the quality of the fit
Five parameters for 72 quotes: the errors are the price of the parsimony. The library's report, then the smiles.

In [ ]:
print(model.calibration.round(3).to_string(index=False))
fig, axes = plt.subplots(2, 4, figsize=(16, 6.4))
for ax, i in zip(axes.ravel(), range(len(T_NODES))):
    T = T_NODES[i]
    kk = np.linspace(K_MKT[i, 0], K_MKT[i, -1], 41)
    ax.plot(K_MKT[i], 100 * IV[i], "ko", ms=4, label="market")
    ax.plot(kk, 100 * heston_smile(P, kk, T), label="Heston")
    ax.set(title=f"T = {T:.2f}", xlabel="k = K / F")
    ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.show()

From 6M the fit is within about one vol point. At 1M and 3M the market skew is much steeper than any Heston skew
compatible with the rest of the surface (section 2): the model misses the short-dated wings by many vol points. The
weights decide where the error goes: with the ATM-vega weights the short maturities count as much as the long ones in
vol terms, but their deep wings have tiny vegas and count little. Note also Feller: the fit pushes $\xi$ up for the
curvature, and the variance reaches zero.

### Step 5: identifiability, which parameters the quotes actually pin down
Near the optimum the objective is a quadratic form in the parameters, with matrix $J^\top J$ ($J$ the Jacobian of the
residuals). Its inverse is, up to a scale, the covariance of the estimated parameters: large correlations mean that
moving two parameters together barely changes the fit.

In [ ]:
J = fit.jac
cov = np.linalg.inv(J.T @ J)
corr = cov / np.sqrt(np.outer(np.diag(cov), np.diag(cov)))
print("correlation of the parameter estimates:")
print(pd.DataFrame(corr, index=NAMES, columns=NAMES).round(2).to_string())

# the same in a picture: refit with kappa fixed at other values, the others free
rows = []
for kap in (0.15, 0.25, 0.36, 0.6, 1.0, 2.0):
    lo, hi = LOWER.copy(), UPPER.copy()
    lo[1], hi[1] = kap * (1 - 1e-9), kap * (1 + 1e-9)
    f = least_squares(residuals, np.clip(np.array(P), lo, hi), bounds=(lo, hi), method="trf", x_scale="jac", jac="2-point")
    rows.append({"kappa (fixed)": kap, **dict(zip(NAMES, f.x)), "rms (vol pts)": 100 * np.sqrt(np.mean(f.fun ** 2))})
print("\nrefit with kappa fixed:")
print(pd.DataFrame(rows).drop(columns="kappa").round(4).to_string(index=False))

$\kappa$, $\theta$ and $\xi$ trade off: a faster reversion with a higher vol of vol and a different long-run level gives a fit
of almost the same quality. The surface pins down combinations (the ATM term structure, the skew at each maturity),
not each parameter. In practice the parameters jump from one day to the next while the fitted surface barely moves;
exotic prices that depend on a single parameter (a forward-start on $\xi$, say) move with them.

### Step 6: sensitivities through the calibration
A greek of a Heston price w.r.t. a market quote $m$ (a vol node, the spot through the forwards) goes through the
parameters: $\frac{\mathrm{d}V}{\mathrm{d}m} = \frac{\partial V}{\partial m} + \frac{\partial V}{\partial p}\,\frac{\mathrm{d}p}{\mathrm{d}m}$. The optimum satisfies $\nabla_p L(p^*, m) = 0$ for every $m$;
differentiating this identity (implicit function theorem),
$$\frac{\mathrm{d}p^*}{\mathrm{d}m} = -\,H^{-1}\,\frac{\partial^2 L}{\partial p\,\partial m},\qquad H = \nabla_p^2 L(p^*, m),$$
the derivative of a re-optimisation without re-optimising. The library computes $H$ exactly (AAD) and writes the
parameters as $p^* - (s - \mathrm{stop\_gradient}(s))$ with $s = H^{-1}\nabla_p L$: the value stays $p^*$, the gradient is the formula above.
Below, $\mathrm{d}p / \mathrm{d}\sigma_{ij}$ per vol node: which quotes move which parameter.

In [ ]:
with tf.GradientTape(persistent=True) as tape:
    m_tape = tq.HestonModel.from_equity(eq)
    params = m_tape.parameters
fig, axes = plt.subplots(1, 5, figsize=(18, 3.4), sharey=True)
for ax, n in zip(axes, NAMES):
    g = tape.gradient(params[n], vs.volatility_matrix).numpy() / 100        # per vol point
    lim = np.abs(g).max()
    im = ax.imshow(g, aspect="auto", cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.set(title=f"d {n} / d vol node", xticks=range(len(K_NODES)), xticklabels=[f"{k:.0%}" for k in K_NODES / S0],
           yticks=range(len(T_NODES)), yticklabels=np.round(T_NODES, 2), xlabel="K / S0")
    ax.tick_params(axis="x", labelrotation=90)
    fig.colorbar(im, ax=ax)
axes[0].set_ylabel("T")
plt.show()

Every parameter depends on many nodes:
- $v_0$ mostly on the short ATM vols;
- $\kappa$ and $\theta$ on the long maturities with opposite patterns (the trade-off of step 5);
- $\rho$ on the skew: raising the downside vols makes it more negative, raising the upside vols less negative;
- $\xi$ on the curvature: it rises with the short ATM vols and the wings, and falls when the 1Y–2Y ATM vols rise (a
  larger $\xi$ lowers the ATM vol).

So a Heston price has **vega on the whole surface**, also on nodes far from its own strike and maturity: the model's
view of how the quotes are linked.

### How Heston is calibrated in practice
- **Objective**: vega-weighted prices (as here) or implied vol errors directly; often only the maturities and strikes
  relevant to the product, since one parameter set cannot fit everything.
- **Local minima**: the objective is not convex; a global search (differential evolution, multi-start) or a good start
  (yesterday's parameters) is common. Some desks fix $v_0$ from the short ATM vol or $\kappa$ by judgement and fit the rest.
- **Stability**: a penalty that keeps the parameters close to yesterday's trades a little fit for stable greeks.
- **Better fit**: time-dependent parameters (piecewise constant, Andersen and Piterbarg, 2010), jumps for the short-dated
  skew (Bates, 1996), or the exact fit of the stochastic local vol model (Heston dynamics with a leverage function).

## 5. What the model implies: the forward smile
A forward-start call pays $(S_{T_2}/S_{T_1} - k)^+$: its implied vol is the smile the model predicts for the period
$[T_1, T_2]$, seen from today. In Heston the smile at $T_1$ is driven by the same $\xi$ and $\rho$ as today's, only the level
depends on the variance $v_{T_1}$ at that date: the **forward smile keeps its skew**. Local vol flattens it.

Below, from 1Y to 2Y, with the same draws for both models: today's 1Y market smile, the two forward smiles, and the
Heston forward smile conditional on the variance at 1Y (low / high third of the paths): in Heston the future smile is
a random object, in local vol it is fixed.

In [ ]:
grid2 = np.arange(1, 2 * 50 + 1) / 50                                  # 50 steps a year to 2Y
dw2 = tf.random.stateless_normal([40_000, len(grid2), 2], seed=[3, 0], dtype=tf.float64)
X2, V2 = (a.numpy() for a in model.simulate(grid2, dw2))
lv = tq.LocalVolatilityModel.from_equity(eq)
S_lv = lv.evolve(grid2, dw2[..., 0]).numpy()
c1y, c2y = 49, 99                                                      # columns of 1Y and 2Y
ratio_h = X2[:, c2y] / X2[:, c1y]                                       # E[ratio] = 1 (X is a martingale)
ratio_lv = (S_lv[:, c2y] / S_lv[:, c1y]) / (float(eq.forward(2.0)) / float(eq.forward(1.0)))   # X_2 / X_1


def mc_smile(ratio, ks, T=1.0):
    return np.array([implied(np.mean(np.maximum(ratio - k, 0.0)), k, T) for k in ks])


ks = np.linspace(0.7, 1.3, 13)
F1 = float(eq.forward(1.0))
today = np.array([float(vs.volatility(k * F1, 1.0)) for k in ks])
low, high = V2[:, c1y] <= np.quantile(V2[:, c1y], 1 / 3), V2[:, c1y] >= np.quantile(V2[:, c1y], 2 / 3)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(ks, 100 * today, "k-o", ms=3, label="today's 1Y smile (market)")
axes[0].plot(ks, 100 * mc_smile(ratio_h, ks), "-s", ms=3, label="Heston forward 1Y->2Y")
axes[0].plot(ks, 100 * mc_smile(ratio_lv, ks), "-^", ms=3, label="local vol forward 1Y->2Y")
axes[0].set(xlabel="k (strike / forward)", ylabel="implied vol (%)", title="Forward smile")
axes[1].plot(ks, 100 * mc_smile(ratio_h[low], ks), "-o", ms=3, label="v at 1Y in the low third")
axes[1].plot(ks, 100 * mc_smile(ratio_h, ks), "-s", ms=3, label="all paths")
axes[1].plot(ks, 100 * mc_smile(ratio_h[high], ks), "-^", ms=3, label="v at 1Y in the high third")
axes[1].set(xlabel="k (strike / forward)", title="Heston forward smile given the variance at 1Y")
for ax in axes:
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
plt.show()
i70, i100 = 0, int(np.argmin(np.abs(ks - 1.0)))
for name, smile in (("market today 1Y", today), ("Heston forward", mc_smile(ratio_h, ks)), ("local vol forward", mc_smile(ratio_lv, ks))):
    print(f"{name:18s}: ATM {100 * smile[i100]:.1f}%, 70% - ATM skew {100 * (smile[i70] - smile[i100]):.1f} vol points")

- **Local vol** flattens the forward smile: the downside skew from 70% to ATM is about half of today's.
- **Heston** keeps it, here even steeper, with more curvature: the same $\xi$ and $\rho$ act on every period. Its forward ATM
  level is lower, for the same reason as the falling ATM term structure (section 2: average vol below the square root of
  the average variance).
- Both models fit today's 1Y and 2Y smiles (Heston approximately), yet they give different prices to a forward-start
  option, a cliquet, or the future barriers of an autocallable. Which one is right is not in the vanilla quotes: it is
  model risk, priced by products that depend on the forward smile.
- In Heston the future smile is a **random object** that depends on the state: after a rise of the variance the whole
  smile is much higher (and flatter in relative terms); after a fall, lower and steeper.

## 6. Monte Carlo

### The QE scheme (Andersen, 2008)
An Euler step $v' = v + \kappa(\theta - v)\Delta + \xi\sqrt{v}\sqrt{\Delta}Z$ can go negative and is biased near zero, which is exactly where the
variance spends time when Feller fails. The quadratic-exponential (QE) scheme draws $v'$ from a distribution that
matches the **exact conditional mean and variance** of the CIR process,
$$m = \theta + (v - \theta)e^{-\kappa\Delta},\qquad s^2 = \frac{v\,\xi^2 e^{-\kappa\Delta}(1 - e^{-\kappa\Delta})}{\kappa} + \frac{\theta\,\xi^2(1 - e^{-\kappa\Delta})^2}{2\kappa},\qquad \psi = s^2 / m^2:$$
- $\psi \le 1.5$ (variance far from zero): $v' = a\,(b + Z)^2$, a squared shifted normal with $a$, $b$ chosen to match $m$, $s^2$;
- $\psi > 1.5$ (variance near zero): a probability mass $p$ at zero and an exponential tail, with $p$ and the tail matched to
  $m$, $s^2$ (drawn by inverting the distribution with $U = N(Z)$).

Then $\ln X$ is stepped exactly given the two variances, with $\int v\,\mathrm{d}t$ approximated by the trapezoid and the correlated part
of $W^1$ written through the variance increment; the drift is chosen so that $\mathbb{E}[X' \mid X] = X$ exactly (the
martingale correction "QE-M"). Two normals per step. QE is accurate with large steps.

### Closed form against simulation
OTM options at 1Y and 3Y, 40k paths, with 4 and 50 steps a year.

In [ ]:
rows = []
for steps in (4, 50):
    g = np.arange(1, 3 * steps + 1) / steps
    d = tf.random.stateless_normal([40_000, len(g), 2], seed=[11, 0], dtype=tf.float64)
    Xs = model.simulate(g, d)[0].numpy()
    for T in (1.0, 3.0):
        x_T = Xs[:, int(round(T * steps)) - 1]
        x_resc = x_T / x_T.mean()                                         # sample mean rescaled to 1, as in pricing
        z_mean = (x_T.mean() - 1) / (x_T.std() / np.sqrt(len(x_T)))
        for k in (0.7, 0.85, 1.0, 1.15, 1.3):
            phi_k = -1.0 if k < 1 else 1.0
            cf = tq.heston_price(np.array([[k]]), [T], *P, phi=phi_k).numpy()[0, 0]
            row = {"steps/yr": steps, "T": T, "k": k, "closed form": cf, "E[X]-1 (s.e.)": z_mean}
            for label, x in (("raw", x_T), ("rescaled", x_resc)):
                payoff = np.maximum(phi_k * (x - k), 0)
                row[f"MC {label}"] = payoff.mean()
                row[f"z {label}"] = (payoff.mean() - cf) / (payoff.std() / np.sqrt(len(payoff)))
            rows.append(row)
print(pd.DataFrame(rows).round(4).to_string(index=False))

- **Raw paths**: the errors are within about 3 standard errors at both step sizes, even with quarterly steps and Feller
  violated: QE does not need a fine grid for European payoffs (path-dependent ones need the grid of their
  observation dates). The errors of the calls and the puts of one maturity move together: call $-$ put $= \mathbb{E}[X_T] - k$, so a
  fluctuation of the sample mean of $X_T$ (column `E[X]-1`, in standard errors) shifts all of them. The step is an exact
  martingale: over many seeds that column averages to zero.
- **Rescaled paths** (sample mean of $X_T$ set to 1, what the pricers do): that common fluctuation is removed, a free
  control variate, and the errors shrink.

### Greeks by AAD
`VanillaMCPricer(model=HestonModel)` calibrates the model inside the tape, so delta and the vega per vol node go
through the calibration (step 6) and through the paths in one backward pass. The closed form gives the same greeks
without Monte Carlo noise. A 1Y ATM call, against Black–Scholes (with the same cash dividends).

In [ ]:
expiry = date_of(1.0)
K = round(float(eq.forward(expiry)), 2)


def call():
    return tq.VanillaOption(eq.currency, start_date=EVALUATION_DATE, end_date=expiry, option_type=tq.OptionType.Call,
                            strike=K, underlying=TICKER)


sources = [eq.spot, vs.volatility_matrix]
with tf.GradientTape() as tape:
    cf_price = tq.HestonModel.from_equity(eq).vanilla_price(K, expiry, 1.0)
d_cf, v_cf = tape.gradient(cf_price, sources)
mc_call, bs_call_ = call(), call()
mc = tq.VanillaMCPricer(model=tq.HestonModel, n_paths=40_000, n_steps=50, seed=7)
mc.price(mc_call, mkt, autodiff=True)
bs = tq.BlackScholesPricer(dividend_model="discrete")
bs.price(bs_call_, mkt, autodiff=True)
(d_mc, v_mc), (d_bs, v_bs) = mc.tape.gradient(mc_call.price, sources), bs.tape.gradient(bs_call_.price, sources)
print(f"price  Heston CF {float(cf_price):.3f}   MC {float(mc_call.price):.3f} +/- {float(mc_call.standard_error):.3f}   "
      f"BS {float(bs_call_.price):.3f}")
print(f"delta  Heston CF {float(d_cf):.4f}   MC {float(d_mc):.4f}   BS {float(d_bs):.4f}")
print(f"vega (sum over nodes, per vol point)  Heston CF {float(tf.reduce_sum(v_cf)) / 100:.3f}   "
      f"MC {float(tf.reduce_sum(v_mc)) / 100:.3f}   BS {float(tf.reduce_sum(v_bs)) / 100:.3f}")
fig, axes = plt.subplots(1, 3, figsize=(16, 3.4), sharey=True)
for ax, v, name in ((axes[0], v_cf, "Heston closed form"), (axes[1], v_mc, "Heston MC (AAD)"), (axes[2], v_bs, "Black-Scholes")):
    im = ax.imshow(v.numpy() / 100, aspect="auto", cmap="RdBu_r", vmin=-0.4, vmax=0.4)
    ax.set(title=f"vega per node: {name}", xticks=range(len(K_NODES)), xticklabels=[f"{k:.0%}" for k in K_NODES / S0],
           yticks=range(len(T_NODES)), yticklabels=np.round(T_NODES, 2), xlabel="K / S0")
axes[0].set_ylabel("T")
fig.colorbar(im, ax=axes)
plt.show()

Black–Scholes puts the whole vega on the 1Y nodes around the strike. Heston spreads it over the surface, with signs:
the price depends on the five parameters, and each of them on many nodes (step 6). The total is close (a parallel
shift of the surface moves the price by about the same amount in both models), the buckets are not. The Heston price
of the ATM call is not exactly the market one either: the calibration error at 1Y.

## 7. The limits of Heston
Heston gives the vol a dynamics of its own, which is what local vol misses, but with five constant parameters.

**1. It does not fit the whole surface.** The short-dated equity skew is too steep for a diffusion with a finite
short-time skew $\rho\xi / (4\sqrt{v_0})$, and the skew decays too fast or too slowly with the maturity (section 2). The fit
errors are largest at the short maturities and in the wings (section 4): vanillas and anything that depends on them
(digitals, barriers near expiry) are mispriced there. Remedies: time-dependent parameters, jumps (Bates), or the
stochastic local vol model, which corrects the marginals exactly with a leverage function.

**2. The parameters are poorly identified.** $\kappa$, $\theta$ and $\xi$ trade off (step 5): close fits with different parameters,
and parameters that jump between calibrations. Prices that depend on a single parameter, and the vega buckets, inherit
the instability.

**3. The dynamics is imposed by today's smile.** The same $\xi$ and $\rho$ that fit today's skew and curvature set the
forward smile and the vol of vol of the paths. Nothing guarantees that the vol of vol that fits the 1Y wings is the one
the market will realise or charges for forward skew; there is no separate dial (the stochastic local vol mixing
factor is one).

**4. Feller is usually violated.** The calibration uses a large $\xi$ for the curvature and the variance touches zero.
The prices are fine, but the simulation needs a scheme built for it (QE: an Euler scheme is biased), and the
distribution of the variance (a mass near zero) is not very realistic.

**5. Constant correlation and mean-reversion level.** The spot–vol correlation and the long-run variance are fixed
numbers; regimes (a crisis with correlation going to $-1$, vol levels that drift) are outside the model.

**6. A local optimisation.** The objective is not convex: the result can depend on the start, and the weights decide
where the error goes. The vega per node is spread over the surface, which makes hedging with the vanillas of the
product's own strike and maturity less direct.

| limit | consequence | remedy |
|---|---|---|
| short-dated skew, term structure of skew | vanillas and near-expiry exotics mispriced | jumps, time-dependent parameters, SLV |
| poorly identified parameters | unstable parameters and buckets | fix some parameters, penalise changes |
| dynamics tied to today's smile | forward smile / vol of vol not controlled | SLV (mixing), models with more factors |
| Feller violated | mass at zero, needs QE | QE scheme (here), or accept a worse fit |
| constant correlation and level | no regimes | richer models |
| local optimisation, spread vegas | start-dependent result, indirect hedge | global search, good start, regularisation |

In short: Heston is the reference model for the **dynamics** of the vol (forward smile, vol of vol) and has a fast
closed form for vanillas; it does not reprice the surface. Local vol reprices the surface but has the wrong dynamics.
The stochastic local vol model takes the dynamics of the first and the marginals of the second.

## Exercises
1. In the hand calibration (step 3), fix $\rho = 0$ with the bounds: what happens to the skew and to the errors?
2. Calibrate only to the maturities from 6M (`residuals(x, rows=slice(2, None))`): how do the parameters and the 1M error
   change? Why can this be the right choice for a 3Y product?
3. Double $\xi$ with the other parameters fixed: what happens to the forward smile of section 5 and to the 1Y wings?